# Clasificacion de Especies Iris
## Proyecto Final - Mineria de Datos
### Universidad de la Costa

## 1. Importar librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

print('Librerias importadas correctamente')

## 2. Cargar el dataset Iris

In [ ]:
iris = load_iris()
datos = iris.data
etiquetas = iris.target

print('Nombres de las especies:', iris.target_names)
print('Nombres de las caracteristicas:', iris.feature_names)
print('Cantidad de muestras:', len(datos))
print('Forma de los datos:', datos.shape)

In [ ]:
# Crear DataFrame
df = pd.DataFrame(datos, columns=iris.feature_names)
df['especie'] = etiquetas

# Renombrar columnas a español para que sea mas facil
df.columns = ['largo_sepalo', 'ancho_sepalo', 'largo_petalo', 'ancho_petalo', 'especie']

# Mapear numeros a nombres
especies_nombres = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
df['especie_nombre'] = df['especie'].map(especies_nombres)

df.head()

## 3. Explorar los datos

In [ ]:
df.describe()

In [ ]:
# Ver cuantas muestras hay de cada especie
df['especie_nombre'].value_counts()

## 4. Visualizar los datos

In [ ]:
# Grafico de dispersion (sepalos)
plt.figure(figsize=(10, 6))
colores = {'Setosa': 'blue', 'Versicolor': 'red', 'Virginica': 'green'}

for especie in ['Setosa', 'Versicolor', 'Virginica']:
    sub = df[df['especie_nombre'] == especie]
    plt.scatter(sub['largo_sepalo'], sub['ancho_sepalo'], 
                c=colores[especie], label=especie, alpha=0.7)

plt.xlabel('Largo del Sepalo (cm)')
plt.ylabel('Ancho del Sepalo (cm)')
plt.title('Sepalo: Largo vs Ancho')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Grafico de dispersion (petalos)
plt.figure(figsize=(10, 6))

for especie in ['Setosa', 'Versicolor', 'Virginica']:
    sub = df[df['especie_nombre'] == especie]
    plt.scatter(sub['largo_petalo'], sub['ancho_petalo'], 
                c=colores[especie], label=especie, alpha=0.7)

plt.xlabel('Largo del Petalo (cm)')
plt.ylabel('Ancho del Petalo (cm)')
plt.title('Petalo: Largo vs Ancho')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Histogramas
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
caracteristicas = ['largo_sepalo', 'ancho_sepalo', 'largo_petalo', 'ancho_petalo']
titulos = ['Largo Sepalo', 'Ancho Sepalo', 'Largo Petalo', 'Ancho Petalo']

for i, (carac, titulo) in enumerate(zip(caracteristicas, titulos)):
    ax = axes[i//2, i%2]
    for especie in ['Setosa', 'Versicolor', 'Virginica']:
        sub = df[df['especie_nombre'] == especie]
        ax.hist(sub[carac], bins=15, alpha=0.5, label=especie, color=colores[especie])
    ax.set_xlabel(titulo + ' (cm)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribucion de ' + titulo)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlacion
plt.figure(figsize=(8, 6))
sns.heatmap(df[caracteristicas].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlacion')
plt.show()

## 5. Dividir en entrenamiento y prueba

In [ ]:
X = df[['largo_sepalo', 'ancho_sepalo', 'largo_petalo', 'ancho_petalo']]
y = df['especie']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print('Datos de entrenamiento:', X_train.shape[0])
print('Datos de prueba:', X_test.shape[0])

## 6. Crear y entrenar el modelo Random Forest

In [ ]:
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

print('Modelo entrenado correctamente')

## 7. Evaluar el modelo

In [ ]:
y_pred = modelo.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print('=== Metricas del Modelo ===')
print(f'Accuracy:  {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1-Score:  {f1:.3f}')

In [ ]:
# Matriz de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.title('Matriz de Confusion')
plt.show()

## 8. Probar con una flor nueva

In [ ]:
# Vamos a inventar una flor y ver que especie predice el modelo
nueva_flor = pd.DataFrame({
    'largo_sepalo': [5.5],
    'ancho_sepalo': [3.0],
    'largo_petalo': [4.0],
    'ancho_petalo': [1.5]
})

pred = modelo.predict(nueva_flor)[0]
prob = modelo.predict_proba(nueva_flor)[0]

print('Medidas de la flor:')
print(f'  Largo sepalo: 5.5 cm')
print(f'  Ancho sepalo: 3.0 cm')
print(f'  Largo petalo: 4.0 cm')
print(f'  Ancho petalo: 1.5 cm')
print()
print('Especie predicha:', especies_nombres[pred])
print()
print('Probabilidades:')
for i, nombre in enumerate(iris.target_names):
    print(f'  {nombre}: {prob[i]*100:.1f}%')

## 9. Conclusion

El modelo Random Forest clasifica correctamente las especies Iris con un accuracy superior al 95%. 

Esto se debe a que las especies son bastante diferenciables por las medidas de sus petalos y sepalos, especialmente el largo y ancho del petalo.